# Signature-only ablation - is it message passing or metric learning?

Stage A dropped `disease_protein` and `disease_disease` from **both** the graph and
`DistMultPredictor`'s disease signature at once, and they came out as by far the two most important
relations in the KG:

```
drop_disease_protein_full   d_indication -0.1042   d_contraindication -0.1483   rank -0.1263
drop_disease_disease_full   d_indication -0.0533   d_contraindication -0.0194   rank -0.0363
drop_drug_protein           d_indication -0.0135   d_contraindication -0.0058   rank -0.0096   <- 3rd place
...every other relation      |rank| <= 0.006
```

That result cannot say *which half* of the ablation did the damage. This notebook separates them: the
relations stay **fully present in message passing** (the graph is untouched, all 50 edge types, all
6,735,310 training edges) and are removed **only from the metric-learning signature**
(`disease_etypes` at `model.py:88`, fallback at `model.py:206`).

If the signature-only delta reproduces most of the Stage A delta, the relation matters because of
metric learning, not because of the GNN. That is the claim this notebook tests.

### Arms (5, x 10 seeds = 50 runs)

| arm | graph | signature `disease_etypes` | `proto` |
|---|---|---|---|
| `baseline` | full | `['disease_disease', 'rev_disease_protein']` | True |
| `sig_drop_disease_protein` | full | `['disease_disease']` | True |
| `sig_drop_disease_disease` | full | `['rev_disease_protein']` | True |
| `sig_drop_both` | full | `[]` -> zero vector | True |
| `proto_off` | full | n/a | **False** |

`sig_drop_both` is the literal "both present in message passing, absent in metric learning" request.
Read it with the caveat in section 4: an empty signature does not cleanly switch metric learning off, it
leaves the `rarity` gate multiplying the disease embedding by `(1 - c_i)`. `proto_off` is the clean
"no metric learning" reference - it is the published TxGNN ablation and skips the block entirely.
Together the two bracket the effect.

### Why the signature patch is load-bearing here and was not in Stage A

In Stage A the relation was also gone from `G`, so `obtain_disease_profile` already returned an all-zero
block for it and removing it from the list changed nothing numerically (cosine similarity is invariant to
a common zero block). Here the edges are still in `G`, so the block is genuinely populated and removing
it from `disease_etypes` genuinely changes the signature. Same patch, opposite significance.

### Resumability

Identical to `ablation_stage_a.ipynb`: results land in `results/ablation_signature_only/<run_key>/` the
moment each run finishes, a run with both output files is skipped, and the notebook is safe to re-run
from cell 1 after a crash, kernel restart or interrupt.

---
## 1 - Environment

In [ ]:
import os, sys, json, time, gc, random, shutil, inspect, platform, warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')


def _find_repo_root(start=None):
    p = Path(start or os.getcwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / 'pyg_implementation' / 'txgnn').is_dir():
            return cand
    raise RuntimeError('Could not find pyg_implementation/txgnn above ' + str(p))


REPO_ROOT = _find_repo_root()
PYG_ROOT  = REPO_ROOT / 'pyg_implementation'

# Put the PyG port FIRST on sys.path so `import txgnn` never resolves to dgl_implementation/.
sys.path.insert(0, str(PYG_ROOT))

import torch
import txgnn
import txgnn.model as txgnn_model
from txgnn import TxData, TxGNN
from txgnn.utils import evaluate_fb, create_pyg_graph, obtain_disease_profile, get_n_params

assert 'pyg_implementation' in txgnn.__file__, (
    'Wrong txgnn package on sys.path (%s). Restart the kernel.' % txgnn.__file__)

# The Gumbel-gate switch was removed from the library in commit 2d064cc. Nothing here passes it;
# fail loudly if it ever comes back, rather than silently training a different model.
_MI_PARAMS = set(inspect.signature(TxGNN.model_initialize).parameters)
assert 'use_decision_network' not in _MI_PARAMS, (
    'model_initialize accepts use_decision_network again - revisit MODEL_CFG before trusting this grid.')


def set_all_seeds(seed: int):
    '''Seed every RNG the training path touches.'''
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'

print('repo root     :', REPO_ROOT)
print('txgnn package :', txgnn.__file__)
print('torch         :', torch.__version__)
print('device        :', DEVICE, '|',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else platform.processor())
if DEVICE == 'cpu':
    print('\nWARNING: CPU only. A ~20 GPU-hour grid is not practical here.')

---
## 2 - Configuration

Model and schedule are identical to `ablation_stage_a.ipynb` so the two grids are directly comparable -
same `full` profile from `pyg_implementation/reproduce/train.py`, same one-epoch (~6,578-step)
pretraining pass, same `train_print_per_n` (logging only, no effect on any number).

`SEEDS` runs to 10. Seeds 6-10 have no split on disk yet, so `prepare_split` will create
`data/complex_disease_<seed>/` for each of them the first time it is asked - a few minutes per seed,
once.

In [ ]:
SPLIT   = 'complex_disease'
SEEDS   = [10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
PROFILE = 'full'

MODEL_CFG = dict(
    n_hid       = 100,
    n_inp       = 100,
    n_out       = 100,
    proto       = True,                  # disease-similarity metric learning (arm proto_off sets False)
    proto_num   = 3,
    attention   = False,
    sim_measure = 'all_nodes_profile',
    agg_measure = 'rarity',
)

PRETRAIN_CFG = dict(n_epoch=1,    learning_rate=1e-3, batch_size=1024, train_print_per_n=500)
FINETUNE_CFG = dict(n_epoch=1000, learning_rate=5e-4, train_print_per_n=100, valid_per_n=20)

DD_REL     = ['contraindication', 'indication', 'off-label use']
DD_REV_REL = ['rev_' + r for r in DD_REL]
DD_ETYPES  = [('drug', r, 'disease') for r in DD_REL] + [('disease', r, 'drug') for r in DD_REV_REL]

# the library default, for reference in every log line and metadata record
SIG_ETYPES_DEFAULT = ['disease_disease', 'rev_disease_protein']
SIG_NODES_DEFAULT  = ['disease', 'gene/protein']

DATA_FOLDER = REPO_ROOT / 'data'
ABL_ROOT    = REPO_ROOT / 'results' / 'ablation_signature_only'
STAGE_A_ROOT = REPO_ROOT / 'results' / 'ablation_stage_a'
ABL_ROOT.mkdir(parents=True, exist_ok=True)

# --- switches ------------------------------------------------------------------------------------
RUN_GRID          = True   # section 6.2 trains; set False to inspect the plan / aggregate only
SAVE_MODELS       = False
IMPORT_BASELINE   = True   # reuse Stage A's baseline_seed1..5 - identical config, see section 5
STOP_AFTER_N_RUNS = None   # e.g. 2 for a short unattended smoke pass over the harness

print('split   :', SPLIT)
print('seeds   :', SEEDS)
print('results :', ABL_ROOT)
print('pretrain:', PRETRAIN_CFG)
print('finetune:', FINETUNE_CFG)

---
## 3 - The arms

In [ ]:
ARMS = [
    {'arm': 'sig_drop_disease_protein', 'sig_drop': ['rev_disease_protein'],        'overrides': {}},
    {'arm': 'sig_drop_disease_disease', 'sig_drop': ['disease_disease'],            'overrides': {}},
    {'arm': 'sig_drop_both',            'sig_drop': list(SIG_ETYPES_DEFAULT),       'overrides': {}},
    {'arm': 'proto_off',                'sig_drop': [],                             'overrides': {'proto': False}},
]

# the relation whose Stage A counterpart this arm isolates - used by the section 7 comparison table
ARM_TO_STAGE_A = {
    'sig_drop_disease_protein': 'drop_disease_protein_full',
    'sig_drop_disease_disease': 'drop_disease_disease_full',
}

ARMS_BY_NAME = {a['arm']: a for a in ARMS}
assert len(ARMS_BY_NAME) == len(ARMS), 'duplicate arm name'

print('%d arms x %d seeds = %d runs\n' % (len(ARMS), len(SEEDS), len(ARMS) * len(SEEDS)))
for a in ARMS:
    sig = [e for e in SIG_ETYPES_DEFAULT if e not in a['sig_drop']]
    cfg = dict(MODEL_CFG, **a['overrides'])
    print('  %-26s proto=%-5s signature=%s'
          % (a['arm'], cfg['proto'], sig if cfg['proto'] else 'n/a (metric learning off)'))

---
## 4 - Mechanics

Two pieces, both monkeypatch-free on the library side.

**1. The signature patch.** `txgnn.model.obtain_disease_profile` is replaced with a wrapper that filters
the ablated entries out of the `(disease_etypes, disease_nodes)` pair. Both call sites - `model.py:104`
in `DistMultPredictor.__init__` and `model.py:214` in the unseen-disease fallback - resolve the name from
the `txgnn.model` module globals at call time, so patching the module attribute covers both. `SIG_DROP`
must be set *before* `model_initialize`, because that is where `sim_all_etypes` is built.

**2. `RunData`.** A `TxData` stand-in that rebuilds `G` from the untouched `df_train` for every run, so
no run inherits a graph another run moved to the GPU or re-embedded. The graph here is never ablated -
`run_one` asserts both `disease_disease` and `rev_disease_protein` are still edge types in `G`, which is
the inverse of Stage A's "it really disappeared" check.

### What `sig_drop_both` actually does, and why `proto_off` is also run

With both entries removed, `keep` is empty and the patch returns an explicit zero vector of the default
dimension. It deliberately does **not** delegate to the unpatched call with the unfiltered lists the way
`ablation_stage_a.ipynb` does: there the relation was also gone from `G`, so the unpatched call returned
zeros anyway, but here the edges are still present and it would rebuild the full signature, making the
arm a silent duplicate of the baseline. `run_one` asserts the probed signature maximum is `0.0` for this
arm, so that failure mode cannot pass unnoticed.

Following the zero vector through `DistMultPredictor.forward`:

- `sim_matrix` floors the norm at `eps=1e-8`, so the similarity matrix is all **zeros** - no NaN.
- `topk` on all-zeros returns arbitrary indices and zero values; `F.normalize(zeros, p=1)` returns zeros,
  so the pooled prototype `out` is the **zero vector**.
- `proto_emb = (1 - c_i) * h_query + c_i * 0`, with `c_i = 0.7*exp(-0.7*deg) + 0.2` from the `rarity`
  gate (`model.py:255-262`).

So the prototype contributes nothing, but the gate still **shrinks the disease embedding by
`(1 - c_i)`** - and `c_i` is ~0.9 for the rare/held-out diseases this split is built from, so the disease
side of the DistMult score is attenuated roughly 10x. `sig_drop_both` is therefore "metric learning with
an uninformative signature", not "metric learning removed". `proto_off` (`proto=False`) removes the block
outright and is the clean reference. Report both; do not report `sig_drop_both` alone as "no metric
learning".

In [ ]:
class RunData:
    '''Minimal TxData stand-in. Graph rebuilt per run; splits never filtered in this notebook.'''

    def __init__(self, base, data_folder, split):
        self.df       = base['df']
        self.df_train = base['df_train']
        self.df_valid = base['df_valid']
        self.df_test  = base['df_test']
        self.G = create_pyg_graph(self.df_train, self.df)
        self.data_folder = str(data_folder)
        self.split = split
        self.disease_eval_idx = None
        self.no_kg = False


# --- signature patch ------------------------------------------------------------------------------
SIG_DROP = set()          # set per run by run_one()
_orig_obtain_disease_profile = obtain_disease_profile


def _obtain_disease_profile_ablated(G, disease, disease_etypes, disease_nodes):
    keep = [(e, n) for e, n in zip(disease_etypes, disease_nodes) if e not in SIG_DROP]
    if not keep:
        # sig_drop_both: no source left -> explicit zero vector of the default dimension.
        # Do NOT delegate with the unfiltered lists here: unlike in ablation_stage_a.ipynb the
        # edges are still in G, so the original would happily rebuild the FULL signature and the
        # arm would be a silent duplicate of the baseline. See section 4.
        return torch.zeros(sum(G[n].num_nodes for n in disease_nodes))
    etypes, nodes = zip(*keep)
    return _orig_obtain_disease_profile(G, disease, list(etypes), list(nodes))


txgnn_model.obtain_disease_profile = _obtain_disease_profile_ablated
assert txgnn_model.obtain_disease_profile is _obtain_disease_profile_ablated


# --- per-seed split cache -------------------------------------------------------------------------
_SEED_CACHE = {}


def get_seed_data(seed):
    '''Load (and cache) the split frames for one seed. Holds one seed at a time - `df` is huge.'''
    if seed in _SEED_CACHE:
        return _SEED_CACHE[seed]
    _SEED_CACHE.clear()
    gc.collect()
    print('[data] loading split seed %d ...' % seed)
    t0 = time.time()
    set_all_seeds(seed)
    d = TxData(data_folder_path=str(DATA_FOLDER))
    d.prepare_split(split=SPLIT, seed=seed, no_kg=False)
    base = {'df': d.df, 'df_train': d.df_train, 'df_valid': d.df_valid, 'df_test': d.df_test}
    del d
    gc.collect()
    _SEED_CACHE[seed] = base
    print('[data] seed %d ready in %.1f s | train edges %d'
          % (seed, time.time() - t0, len(base['df_train'])))
    return base


def run_dir(arm, seed):
    return ABL_ROOT / ('%s_seed%d' % (arm, seed))


def is_done(arm, seed):
    d = run_dir(arm, seed)
    return (d / 'edge_level_test_metrics.csv').exists() and (d / 'run_meta.json').exists()


def signature_probe(model):
    '''Read back the signature actually built, as a cheap check the patch took effect.

    Returns (dim, max_value). max_value == 0 means every disease signature is the zero vector.
    '''
    try:
        prof = model.pred.diseases_profile_etypes
        et = next(iter(prof))
        vals = list(prof[et].values())[:64]
        if not vals:
            return None, None
        return int(vals[0].shape[0]), float(max(float(v.max()) for v in vals))
    except Exception as exc:                      # proto=False has no profiles at all
        return None, None

---
## 5 - Plan

`baseline` is configuration-identical to Stage A's `baseline` arm - same split, same model, same
schedule. With `IMPORT_BASELINE = True`, seeds 1-5 are copied from
`results/ablation_stage_a/baseline_seed<n>/` instead of retrained, after checking that the recorded
`split`, `model_config` (the eight keys this notebook sets), `pretrain`, `finetune`,
`dropped_edges_total == 0` and `signature_disease_etypes` all match. Any mismatch falls through to a
normal run. Imported records keep an `imported_from` field, so nothing is silently borrowed.

That saves ~2 GPU-hours. Seeds 6-10 of the baseline are trained here.

In [ ]:
MODEL_KEYS_CHECKED = sorted(MODEL_CFG)       # the keys this notebook actually sets


def import_matching_run(src, dst, expect, extra=None):
    '''Copy a finished run from another grid iff its metadata matches `expect`. Returns (ok, why).'''
    sm, se = src / 'run_meta.json', src / 'edge_level_test_metrics.csv'
    if not (sm.exists() and se.exists()):
        return False, 'no run at %s' % src.name
    meta = json.loads(sm.read_text())
    for k, want in expect.items():
        got = meta.get(k, '<missing>')
        if isinstance(want, dict):
            if not isinstance(got, dict):
                return False, '%s is not a dict' % k
            bad = {kk: (got.get(kk, '<missing>'), vv) for kk, vv in want.items()
                   if got.get(kk, '<missing>') != vv}
            if bad:
                return False, '%s mismatch %s' % (k, bad)
        elif got != want:
            return False, '%s mismatch: %r != %r' % (k, got, want)

    dst.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(se, dst / 'edge_level_test_metrics.csv')
    meta.update(extra or {})
    meta['imported_from'] = str(src.relative_to(REPO_ROOT))
    meta['run_key'] = dst.name
    meta['arm'] = dst.name.rsplit('_seed', 1)[0]
    with open(dst / 'run_meta.json', 'w') as f:
        json.dump(meta, f, indent=2, default=str)
    return True, 'imported from %s' % src.relative_to(REPO_ROOT)


def try_import_baseline(seed):
    if not IMPORT_BASELINE:
        return False, 'disabled'
    return import_matching_run(
        STAGE_A_ROOT / ('baseline_seed%d' % seed),
        run_dir('baseline', seed),
        expect={'split': SPLIT, 'seed': seed,
                'dropped_edges_total': 0,
                'signature_disease_etypes': SIG_ETYPES_DEFAULT,
                'model_config': {k: MODEL_CFG[k] for k in MODEL_KEYS_CHECKED},
                'pretrain': PRETRAIN_CFG, 'finetune': FINETUNE_CFG},
        extra={'proto': True, 'signature_sig_drop': [], 'message_passing_intact': True},
    )


# --- run the import pass now, so the plan below reflects it ---------------------------------------
if IMPORT_BASELINE:
    for s in SEEDS:
        if is_done('baseline', s):
            continue
        ok, why = try_import_baseline(s)
        print('[import] baseline_seed%d: %s' % (s, why if ok else 'will train (%s)' % why))

BASE_PRETRAIN_S, BASE_FINETUNE_S = 1155.7, 316.5
_ref = STAGE_A_ROOT / 'baseline_seed1' / 'run_meta.json'
if _ref.exists():
    _r = json.loads(_ref.read_text())
    BASE_PRETRAIN_S = _r.get('pretrain_seconds', BASE_PRETRAIN_S)
    BASE_FINETUNE_S = _r.get('finetune_seconds', BASE_FINETUNE_S)
EST_MIN = (BASE_PRETRAIN_S + BASE_FINETUNE_S) / 60

rows = []
for a in ARMS:
    done = [s for s in SEEDS if is_done(a['arm'], s)]
    rows.append({'arm': a['arm'],
                 'proto': dict(MODEL_CFG, **a['overrides'])['proto'],
                 'signature': ', '.join(e for e in SIG_ETYPES_DEFAULT if e not in a['sig_drop']) or '(empty)',
                 'done': len(done), 'todo': len(SEEDS) - len(done)})
plan = pd.DataFrame(rows)
display(plan.style.hide(axis='index'))

RUN_KEYS = [(a['arm'], s) for s in SEEDS for a in ARMS]      # seed-major, matches the run order
pending  = [k for k in RUN_KEYS if not is_done(*k)]
print('\nrun_keys (%d total, %d done, %d pending):' % (len(RUN_KEYS), len(RUN_KEYS) - len(pending), len(pending)))
for arm, seed in RUN_KEYS:
    print('  [%s] %s_seed%d' % ('x' if is_done(arm, seed) else ' ', arm, seed))
print('\nestimated remaining: %.1f GPU-hours at %.1f min/run' % (len(pending) * EST_MIN / 60, EST_MIN))
print('order: seed-major - after seed %d finishes, every arm has n=1 and can already be read.' % SEEDS[0])

---
## 6 - The grid

Each finished run writes `edge_level_test_metrics.csv` and `run_meta.json` into
`results/ablation_signature_only/<run_key>/`. The metadata records the signature actually used, the
probed signature dimension and maximum (0 means every signature is the zero vector), `proto`, parameter
count, and `sorted(G.edge_types)` - which should be the full 50 for every arm in this notebook.

`MACRO (all relations)` **is** comparable across arms here, unlike in Stage A: the graph is identical in
every arm, so the mean is taken over the same 50 edge types every time.

In [ ]:
def run_one(arm_name, seed, verbose=True):
    '''Train one (arm, seed) and write its results. Returns the edge-level DataFrame.'''
    global SIG_DROP

    arm  = ARMS_BY_NAME[arm_name]
    rdir = run_dir(arm_name, seed)
    el_csv, meta_json = rdir / 'edge_level_test_metrics.csv', rdir / 'run_meta.json'

    if el_csv.exists() and meta_json.exists():
        if verbose:
            print('[skip] %s_seed%d -> cached' % (arm_name, seed))
        return pd.read_csv(el_csv)

    if arm_name == 'baseline':
        ok, why = try_import_baseline(seed)
        if ok:
            print('[skip] baseline_seed%d -> %s' % (seed, why))
            return pd.read_csv(el_csv)

    cfg = dict(MODEL_CFG, **arm['overrides'])
    sig_etypes = [e for e in SIG_ETYPES_DEFAULT if e not in arm['sig_drop']]

    print('\n' + '#' * 78)
    print('# %s   seed %d   proto=%s   signature=%s'
          % (arm_name, seed, cfg['proto'], sig_etypes if cfg['proto'] else 'n/a'))
    print('#' * 78)

    base = get_seed_data(seed)

    set_all_seeds(seed)
    d = RunData(base, DATA_FOLDER, SPLIT)
    etypes_after = sorted('%s|%s|%s' % et for et in d.G.edge_types)

    # inverse of Stage A's check: message passing must be INTACT for both relations
    rel_names = {et[1] for et in d.G.edge_types}
    missing = [r for r in SIG_ETYPES_DEFAULT if r not in rel_names]
    assert not missing, 'message passing is not intact, missing from G: %s' % missing
    assert 'disease_protein' in rel_names, 'forward disease_protein missing from G'

    # signature ablation - must be set before model_initialize, which builds sim_all_etypes
    SIG_DROP = set(arm['sig_drop'])

    print('[graph] train edges %d | edge types %d | disease_disease + disease_protein both present'
          % (len(d.df_train), len(etypes_after)))

    set_all_seeds(seed)
    m = TxGNN(data=d, weight_bias_track=False, exp_name='%s_seed%d' % (arm_name, seed), device=DEVICE)
    m.model_initialize(**cfg)

    sig_dim, sig_max = signature_probe(m.model)
    n_params = get_n_params(m.model)
    print('[model] params %d | signature dim %s | signature max %s'
          % (n_params, sig_dim, sig_max))
    if cfg['proto'] and arm['sig_drop'] == list(SIG_ETYPES_DEFAULT):
        assert sig_max == 0.0, 'sig_drop_both should give an all-zero signature, got max %s' % sig_max

    t0 = time.time()
    set_all_seeds(seed)
    m.pretrain(**PRETRAIN_CFG)
    pre_s = time.time() - t0

    t0 = time.time()
    set_all_seeds(seed)
    m.finetune(**FINETUNE_CFG)
    fin_s = time.time() - t0

    # ---- edge-level test metrics ----
    (auroc_rel, auprc_rel, micro_auroc, micro_auprc, macro_auroc, macro_auprc), test_loss = \
        evaluate_fb(m.best_model, m.g_test_pos, m.g_test_neg, m.G, m.dd_etypes, DEVICE, mode='test')

    el = pd.DataFrame(
        [('%s --%s--> %s' % et, auroc_rel.get(et, np.nan), auprc_rel.get(et, np.nan))
         for et in DD_ETYPES],
        columns=['relation', 'AUROC', 'AUPRC'])
    el.loc[len(el)] = ['MICRO (drug-disease)', micro_auroc, micro_auprc]
    el.loc[len(el)] = ['MACRO (drug-disease)',
                       float(np.nanmean([auroc_rel.get(et, np.nan) for et in DD_ETYPES])),
                       float(np.nanmean([auprc_rel.get(et, np.nan) for et in DD_ETYPES]))]
    el.loc[len(el)] = ['MACRO (all relations)', macro_auroc, macro_auprc]

    rdir.mkdir(parents=True, exist_ok=True)
    el.to_csv(el_csv, index=False)

    meta = {
        'run_key': '%s_seed%d' % (arm_name, seed),
        'arm': arm_name, 'seed': seed, 'split': SPLIT, 'profile': PROFILE,
        'experiment': 'signature_only',
        'message_passing_intact': True,
        'dropped_edges_total': 0,
        'train_edges': len(d.df_train),
        'n_edge_types_after': len(etypes_after),
        'edge_types_after': etypes_after,
        'proto': cfg['proto'],
        'signature_sig_drop': arm['sig_drop'],
        'signature_disease_etypes': sig_etypes,
        'signature_disease_etypes_default': SIG_ETYPES_DEFAULT,
        'signature_dim': sig_dim,
        'signature_max': sig_max,
        'n_params': n_params,
        'model_config': m.config,
        'pretrain': PRETRAIN_CFG, 'finetune': FINETUNE_CFG,
        'pretrain_seconds': pre_s, 'finetune_seconds': fin_s,
        'test_bce_loss': float(test_loss),
        'torch': torch.__version__,
        'device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
        'finished_at': time.strftime('%Y-%m-%dT%H:%M:%S'),
    }
    # written last - is_done() needs both files, so a crash mid-write leaves the run pending
    with open(meta_json, 'w') as f:
        json.dump(meta, f, indent=2, default=str)

    if SAVE_MODELS:
        mdir = REPO_ROOT / 'saved_models' / 'ablation_signature_only' / meta['run_key']
        mdir.mkdir(parents=True, exist_ok=True)
        m.save_model(str(mdir))

    ind = auprc_rel.get(('drug', 'indication', 'disease'), np.nan)
    con = auprc_rel.get(('drug', 'contraindication', 'disease'), np.nan)
    print('[done] %s_seed%d | %.1f min | indication AUPRC %.4f | contraindication AUPRC %.4f -> %s'
          % (arm_name, seed, (pre_s + fin_s) / 60, ind, con, rdir.relative_to(REPO_ROOT)))

    # ---- free the GPU before the next run ----
    SIG_DROP = set()
    del m, d, auroc_rel, auprc_rel
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return el

In [ ]:
if RUN_GRID:
    todo = [(a['arm'], s) for s in SEEDS for a in ARMS if not is_done(a['arm'], s)]
    if STOP_AFTER_N_RUNS:
        todo = todo[:STOP_AFTER_N_RUNS]

    print('%d run(s) pending of %d total.' % (len(todo), len(ARMS) * len(SEEDS)))
    grid_t0 = time.time()
    for i, (arm_name, seed) in enumerate(todo, 1):
        print('\n>>> %d/%d  %s_seed%d  (elapsed %.1f h)'
              % (i, len(todo), arm_name, seed, (time.time() - grid_t0) / 3600))
        run_one(arm_name, seed)
    print('\nGrid finished %d run(s) in %.1f h.' % (len(todo), (time.time() - grid_t0) / 3600))
else:
    print('RUN_GRID is False - no training. Set it in section 2.')

---
## 7 - Aggregate

Same statistics as Stage A: absolute mean AUPRC with a 95% normal CI, plus the **paired** `arm - baseline`
delta computed per seed and then averaged, which removes split-to-split variance.

In [ ]:
def _load_all(root):
    recs = []
    for meta_p in sorted(Path(root).glob('*/run_meta.json')):
        el_p = meta_p.parent / 'edge_level_test_metrics.csv'
        if not el_p.exists():
            continue
        meta = json.loads(meta_p.read_text())
        el = pd.read_csv(el_p).set_index('relation')
        for rel, row in el.iterrows():
            recs.append({'arm': meta['arm'], 'seed': meta['seed'], 'relation': rel,
                         'AUROC': row['AUROC'], 'AUPRC': row['AUPRC'],
                         'proto': meta.get('proto'), 'signature_dim': meta.get('signature_dim'),
                         'imported': 'imported_from' in meta})
    return pd.DataFrame(recs)


def _ci95(x):
    x = np.asarray(x, dtype=float)
    return 1.96 * x.std(ddof=1) / np.sqrt(len(x)) if len(x) > 1 else np.nan


ROW = {'indication': 'drug --indication--> disease',
       'contraindication': 'drug --contraindication--> disease',
       'micro': 'MICRO (drug-disease)'}

long = _load_all(ABL_ROOT)
summary, wide, base = None, None, None

if long.empty:
    print('No runs on disk yet - run the grid cell in section 6 first.')
else:
    wide = (long[long.relation.isin(ROW.values())]
            .pivot_table(index=['arm', 'seed'], columns='relation', values='AUPRC')
            .rename(columns={v: k for k, v in ROW.items()}))
    info = long.groupby('arm')[['proto', 'signature_dim']].first()

    if 'baseline' in wide.index.get_level_values('arm'):
        base = wide.xs('baseline', level='arm')
    else:
        print('[warn] no baseline runs yet - deltas will be NaN.')

    out = []
    for arm, g in wide.groupby(level='arm'):
        g = g.droplevel('arm')
        rec = {'arm': arm, 'n_seeds': len(g),
               'proto': info.loc[arm, 'proto'], 'signature_dim': info.loc[arm, 'signature_dim']}
        for key in ROW:
            rec['%s_AUPRC' % key] = g[key].mean()
            rec['%s_CI' % key] = _ci95(g[key])
            if arm == 'baseline':
                rec['%s_delta' % key], rec['%s_delta_CI' % key] = 0.0, np.nan
            elif base is not None:
                paired = (g[key] - base[key]).dropna()
                rec['%s_delta' % key] = paired.mean() if len(paired) else np.nan
                rec['%s_delta_CI' % key] = _ci95(paired)
                rec['%s_n_paired' % key] = len(paired)
            else:
                rec['%s_delta' % key], rec['%s_delta_CI' % key] = np.nan, np.nan
        rec['rank_score'] = np.nanmean([rec['indication_delta'], rec['contraindication_delta']])
        out.append(rec)

    summary = pd.DataFrame(out).sort_values('rank_score', na_position='last').reset_index(drop=True)
    summary.to_csv(ABL_ROOT / 'signature_only_summary.csv', index=False)

    cols = ['arm', 'n_seeds', 'proto', 'signature_dim',
            'indication_AUPRC', 'indication_CI', 'indication_delta', 'indication_delta_CI',
            'contraindication_AUPRC', 'contraindication_CI', 'contraindication_delta',
            'contraindication_delta_CI', 'micro_AUPRC', 'micro_delta', 'rank_score']
    print('%d arms, %d runs on disk -> %s\n'
          % (len(summary), len(wide), (ABL_ROOT / 'signature_only_summary.csv').relative_to(REPO_ROOT)))
    display(summary[cols].style.format('{:.4f}', subset=cols[4:], na_rep='-').hide(axis='index'))

    short = summary[summary.n_seeds < len(SEEDS)]
    if len(short):
        print('\n[partial] below %d seeds, deltas provisional: %s'
              % (len(SEEDS), ', '.join('%s (n=%d)' % (r.arm, r.n_seeds) for r in short.itertuples())))

In [ ]:
# Per-seed paired deltas - the sign-consistency check a mean hides.
if summary is not None and base is not None:
    for key in ['indication', 'contraindication']:
        arms = [a for a in summary.arm if a != 'baseline']
        tbl = pd.DataFrame({a: (wide.xs(a, level='arm')[key] - base[key]) for a in arms}).T
        seeds_present = list(tbl.columns)
        tbl.columns = ['seed %d' % s for s in seeds_present]
        tbl['mean'] = tbl.mean(axis=1)
        tbl['losses'] = (tbl[['seed %d' % s for s in seeds_present]] < 0).sum(axis=1)
        print('\npaired delta AUPRC vs baseline - %s  (negative = the ablation hurt)' % key)
        display(tbl.style.format('{:.4f}', subset=[c for c in tbl.columns if c != 'losses'], na_rep='-'))

### 7.1 - The headline: how much of Stage A was metric learning?

For `disease_protein` and `disease_disease`, compares three numbers per relation:

- **Stage A** `drop_<rel>_full` - removed from the graph *and* the signature (5 seeds).
- **this notebook** `sig_drop_<rel>` - removed from the signature only, graph intact (10 seeds).
- the same signature-only delta restricted to **seeds 1-5**, so it is an exactly matched comparison
  against Stage A rather than one across different seed sets.

`share_from_signature` is signature-only / Stage A. Near 1.0 means message passing contributed nothing
and the relation mattered purely through metric learning. Above 1.0 means removing it from the signature
alone hurt *more* than removing it from both, which is possible - keeping the edges in the graph while
denying them to the signature is not a subset of the Stage A arm.

In [ ]:
sa_csv = STAGE_A_ROOT / 'stage_a_summary.csv'
if summary is None:
    print('No local runs yet.')
elif not sa_csv.exists():
    print('Stage A summary not found at %s - run that notebook\'s aggregate cell.' % sa_csv)
else:
    sa = pd.read_csv(sa_csv).set_index('arm')
    # the seeds Stage A actually ran, read off its own run records rather than assumed
    sa_seeds = sorted(json.loads(p.read_text())['seed']
                      for p in STAGE_A_ROOT.glob('baseline_seed*/run_meta.json'))
    print('Stage A baseline seeds: %s' % sa_seeds)

    rows = []
    for arm, sa_arm in ARM_TO_STAGE_A.items():
        if arm not in summary.arm.values or sa_arm not in sa.index:
            continue
        s = summary.set_index('arm').loc[arm]
        for key in ['indication', 'contraindication']:
            full = sa.loc[sa_arm, '%s_delta' % key]
            sig  = s['%s_delta' % key]
            # same delta, restricted to the seeds Stage A actually used
            if base is not None:
                g5 = wide.xs(arm, level='arm')[key].reindex(sa_seeds)
                b5 = base[key].reindex(sa_seeds)
                sig5 = (g5 - b5).dropna()
                sig5_mean, sig5_n = (sig5.mean(), len(sig5)) if len(sig5) else (np.nan, 0)
            else:
                sig5_mean, sig5_n = np.nan, 0
            rows.append({
                'relation': arm.replace('sig_drop_', ''), 'metric': key,
                'stage A delta (graph+signature, n=%d)' % int(sa.loc[sa_arm, '%s_n_paired' % key]): full,
                'signature-only delta (n=%d)' % int(s['n_seeds']): sig,
                'signature-only delta (Stage A seeds, n=%d)' % sig5_n: sig5_mean,
                'share_from_signature': sig / full if full else np.nan,
            })

    comp = pd.DataFrame(rows)
    if comp.empty:
        print('Not enough arms finished yet for the comparison.')
    else:
        comp.to_csv(ABL_ROOT / 'vs_stage_a.csv', index=False)
        num = [c for c in comp.columns if c not in ('relation', 'metric')]
        display(comp.style.format('{:.4f}', subset=num, na_rep='-').hide(axis='index'))
        print('\n-> %s' % (ABL_ROOT / 'vs_stage_a.csv').relative_to(REPO_ROOT))

---
## 8 - Reading the result

The claim this notebook can support, if `share_from_signature` for `disease_protein` comes out near 1:

> Removing `disease_protein` from the KG costs TxGNN ~0.10 indication AUPRC, and essentially all of that
> cost is incurred through the disease-similarity signature rather than through message passing. The
> relation's edges can stay in the graph; what matters is that the model can read them when choosing which
> diseases to pool.

Three things it cannot support on its own, and what each needs:

1. **"The GNN contributes nothing."** This notebook varies the signature, not the encoder. The companion
   notebook `ablation_encoder_swap.ipynb` varies the encoder with the signature held fixed; the two
   together make that argument, and `proto_off` here plus `mlp_no_proto` there complete the 2x2.
2. **"Metric learning is the whole model."** `sig_drop_both` is attenuated, not switched off (section 4).
   `proto_off` is the clean floor - quote that one.
3. **Generalisation beyond `complex_disease`.** All of this is one split protocol. Disease-area splits
   would need the zero-shot filtering described at the bottom of `ablation_stage_a.ipynb`.